# 03 先设计验证，再解读真实结果

合成实验知道真值，实景实验首先要回答：**两份结果比较的是不是同一块地、同一种物理量、同一批有效样本？** 我先固定比较单位，再看端元改变、空间聚合和质量筛选分别影响了什么。这样读者可以判断某个数字能够回答哪个问题。

输入是公开 AVIRIS、MOD09GA 与开放验证资料的教学派生成果。AVIRIS 为 2013-05-03 L2 反射率，原生约 14.3 m，处理到 30 m；雪库采用 22°太阳角设定。真实 MODIS 为同日 MOD09GA C6.1，模型设定约 25°；课程 60°模拟见 Notebook 01。WorldView/Landsat 为两场独立覆盖对照，2016 同日，2017 相差一天。

本页首行应打印 `教学数据：62,908 个有效 30 m 像元；420 个候选唯一 1 km 单元`。应出现共同范围地图、固定半径的系数/RMS/残差图、五条候选影像雪谱、420→100→17 筛选图、合成混淆矩阵及独立验证分层图。先学会区分 30 m 像元、500 m 配对和 1 km 状态单元，再解释它们的统计。

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from snowmix import teaching as t
import json
from snowmix.case_analysis import binary_agreement
case = t.load_aviris_case()
units = pd.read_csv(t.demo_directory() / "modis_1km_units.csv")
print(f"教学数据：{int(case['valid_mask'].sum()):,} 个有效 30 m 像元；{len(units)} 个候选唯一 1 km 单元")
print("绘图标签：" + ("中文" if t.configure_plots() else "English（未安装支持的中文字体）"))

## 1 固定空间范围后再比较模型

地图的输入是 `(row, column)` 图层与显式 CRS/transform，输出在同一 UTM 11N 范围显示。两幅雪覆盖图固定 0–1 色标；负值和超过 1 的无约束结果用两端延伸色表示，灰色为 NoData。差值图逐像元相减，因此比单独观察两幅图更容易定位谱库选择的影响。

物理雪库包含 108 个半径候选，可输出光学等效粒径。五种影像雪谱来自不同吸收特征的像元，输出的是谱编号；本案例没有把吸收面积标定为半径。地图中的端元类别用于表示预先选择的光谱类型。

In [ ]:
case = t.load_aviris_case()
spatial = json.loads((t.demo_directory() / "spatial_metadata.json").read_text(encoding="utf-8"))
assert case["representative_spectra"].shape == (4, 182)
assert case["snow_spectra"].shape == (108, 182)
assert case["backgrounds"].shape == (182, 3)
print("有效 30 m 像元：", int(case["valid_mask"].sum()))
figure = t.plot_spatial_case(case, spatial)
display(figure); plt.close(figure)

## 2 一次只改雪粒半径

全区模型差异可能来自端元、波段或样本变化。我先固定四条观测、三个背景谱、182 个有效波段、太阳角和 MGS 求解器，只把半径设为 50、100、200、500、1000 μm。输入数组的轴为 `(4 pixels, 182 bands)`，输出的原始系数为 `(5 radii, 4 pixels, 4 components)`。

上排分别看归一化雪比例和 RMS，下排把雪优势像元的观测、重建与残差并列。若 RMS 降低而雪比例出现非单调变化，要结合背景端元的系数补偿解释；覆盖比例是否更准确还需额外参考。图上保留实际计算的弯曲和转折。

In [ ]:
scan = t.scan_representatives(case)
rows, columns = case["representative_row_col"].T
np.testing.assert_allclose(scan["normalized"][:, :, 0], case["fixed_snow_fraction"][:, rows, columns], atol=2e-6, rtol=2e-5, equal_nan=True)
np.testing.assert_allclose(scan["rms"], np.sqrt(np.mean(scan["residual"]**2, axis=2)), atol=1e-12)
display(t.representative_table(scan).round(6))
figure = t.plot_representative_scan(case, scan)
display(figure); plt.close(figure)

### 全区共同有效集合

原始系数和 RMS 的共同观测集合为 62,908 个像元。覆盖比例比较进一步要求所有候选模型的阴影归一化分母都有效：五个固定半径共享 60,949 个像元，三种端元方法共享 62,861 个像元。下面直接由空间数组重算均值，核对公开 CSV。

In [ ]:
common = case["common_normalization_mask"]
assert int(common.sum()) == 60949
fixed_rows = []
for i, radius in enumerate(case["fixed_radii_um"]):
    fixed_rows.append({"radius_um": radius, "common_n": int(common.sum()),
                       "raw_snow_mean": float(np.mean(case["fixed_raw_snow"][i][common])),
                       "normalized_snow_mean": float(np.mean(case["fixed_snow_fraction"][i][common])),
                       "rms_mean": float(np.mean(case["fixed_rms"][i][common]))})
fixed_common = pd.DataFrame(fixed_rows)
frozen_fixed = pd.read_csv(t.demo_directory() / "fixed_radius_summary.csv")
np.testing.assert_array_equal(fixed_common.common_n, frozen_fixed.common_normalized_n)
for column in ["raw_snow_mean", "normalized_snow_mean", "rms_mean"]:
    np.testing.assert_allclose(fixed_common[column], frozen_fixed["common_"+column], atol=5e-7)
display(fixed_common.round(6))

model_common = case["model_comparison_common_normalization_mask"]
assert int(model_common.sum()) == 62861
model_rows = []
for name, key in [("physical_library_MGS", "physical_snow_fraction"),
                  ("five_image_spectra_MGS", "image_snow_fraction"),
                  ("physical_library_FCLS", "fcls_snow_fraction")]:
    model_rows.append({"method": name, "common_n": int(model_common.sum()),
                       "normalized_snow_mean": float(np.mean(case[key][model_common]))})
model_means = pd.DataFrame(model_rows)
frozen_models = pd.read_csv(t.demo_directory() / "image_endmember_summary.csv")
joined = model_means.merge(frozen_models, on="method", validate="one_to_one", suffixes=("_recomputed", "_frozen"))
np.testing.assert_allclose(joined.normalized_snow_mean_recomputed, joined.common_normalized_snow_mean, atol=5e-7)
display(model_means.round(6))

## 3 用影像雪谱检查谱库选择

物理雪库控制了半径，但实景还含照明、粗糙度和混合效应。因此加入影像雪谱分支，检查“换一组来自现场影像的雪谱”会怎样影响拟合。预先筛选 `NDSI>0.7`、`|NDVI|<0.15`、绿光反射率 0.65–1.1、近红外反射率 >0.11，固定规则后再拟合。

对 950–1090 nm 的 `1−R/C` 做梯形积分，`C` 为端点间直线连续统。输出吸收面积的单位为 μm；用 5%、25%、50%、75%、95% 分位附近的不同像元，得到五种谱型。下面由五条谱重算吸收面积并核对保存值。不同谱型可减少特定雪谱失配，但没有独立粒径测量时仍按谱编号解释。

In [ ]:
from snowmix.case_analysis import absorption_area
area = absorption_area(case["wavelength_um"], case["image_snow_spectra"])
np.testing.assert_allclose(area, case["image_absorption_area_um"], atol=1e-12)
fig, ax = plt.subplots(figsize=(8.6, 4))
for i, (spectrum, color) in enumerate(zip(case["image_snow_spectra"], t.COLORS)):
    t.plot_spectrum(ax, case["wavelength_um"], spectrum, color=color, label=f"S{i+1}: A={area[i]:.4f} μm")
ax.set(xlabel=t.label("波长 / μm", "Wavelength / μm"), ylabel=t.label("反射率", "Reflectance"))
ax.legend(fontsize=8); fig.tight_layout(); display(fig); plt.close(fig)
display(pd.DataFrame({"spectrum_id": np.arange(1, 6), "row": case["image_endmember_row_col"][:, 0], "column": case["image_endmember_row_col"][:, 1], "absorption_area_um": area}))

## 4 验证设计：面积支持怎样改变样本结构

MOD09GA `state_1km` 的 bit 12 为 MOD35 snow/ice，bit 15 为 internal snow mask。它们原生属于约 1 km 单元；把标志复制到 500 m 后会产生重复记录，因此此处按 `roi + tile_row_1km + tile_column_1km` 唯一计数。高分辨率模型按真实相交面积聚合到这个单元，保留有效面积比例。

先从 420 个候选单元建立账本，检查状态值，再要求 AVIRIS 有效面积 ≥95%，最后要求 MODIS 模型有效面积也 ≥95%。四级账本来自同一套键，后一级始终是前一级的子集。图左只画发生样本减少的 420→100→17；图右另画阳性数量，防止把“没有阳性”读成“没有样本”。

MODIS 有效输出同时受影像 QA、AVIRIS 输入支持和归一化条件影响，账本说明的是这些条件的联合筛选结果。此前报告的 85 个历史 500 m 配对属于另一模型和掩膜，单独保留在尺度分析中。

In [ ]:
ledger = t.validation_ledger(units)
np.testing.assert_array_equal(ledger.n, [420, 420, 100, 17])
np.testing.assert_array_equal(ledger.flag12_positive, [32, 32, 0, 0])
np.testing.assert_array_equal(ledger.flag15_positive, [119, 119, 20, 0])
# 直接从同一张唯一单元表重算末级掩膜，核对保存的标记。
common_mask = (units.state_1km != 65535) & (units.aviris_valid_area_fraction >= .95) & (units.modis_valid_area_fraction >= .95)
np.testing.assert_array_equal(common_mask, units.common_valid)
display(ledger)
figure = t.plot_validation_design(ledger)
display(figure); plt.close(figure)

保留 17 个单元后，两种标志均无阳性。它们可以用于观察阴性一致性；参照召回率的分母为零，应为空。最初 119 个 bit 15 阳性全部落在被筛除的支持范围内，这说明未来验证需要在设计阶段保证雪区也有充分的共同面积支持。

### 用一个明确的合成例子学习混淆矩阵

以下 8 个数完全用于教学，包含两类参照。以阈值 0.5 把覆盖比例转成预测类别；行表示参照，列表示预测。手算应得到 TP=2、FN=2、FP=1、TN=3，召回率 0.5、特异度 0.75。改变阈值时先写出哪几个样本会换类，再运行代码。

In [ ]:
synthetic_flag = np.array([1, 1, 1, 1, 0, 0, 0, 0])
synthetic_fraction = np.array([.9, .7, .2, .05, .8, .4, .1, 0])
synthetic = binary_agreement(synthetic_flag, synthetic_fraction, threshold=.5)
assert (synthetic["tp"], synthetic["fn"], synthetic["fp"], synthetic["tn"]) == (2, 2, 1, 3)
assert synthetic["recall"] == .5 and synthetic["specificity"] == .75
matrix = pd.DataFrame([[synthetic["tp"], synthetic["fn"]], [synthetic["fp"], synthetic["tn"]]],
                      index=["Reference snow", "Reference no snow"], columns=["Predicted snow", "Predicted no snow"])
display(matrix)
print("合成示例：recall = 0.50；specificity = 0.75")

现在把相同函数用于真实表。阈值 0.1 是预设主值，0.5 与 0.9 检查阈值敏感性。观测单位与参照类别已明确，才能判断每列指标的分母。下面同时保留 bit 12、bit 15，避免把两种标志混成一个产品。

In [ ]:
units = pd.read_csv(t.demo_directory() / "modis_1km_units.csv")
assert not units.duplicated(["roi", "tile_row_1km", "tile_column_1km"]).any()
agreement = t.product_consistency(units)
assert np.array_equal(units.flag12.to_numpy(), ((units.state_1km.to_numpy(np.uint16) >> 12) & 1))
assert np.array_equal(units.flag15.to_numpy(), ((units.state_1km.to_numpy(np.uint16) >> 15) & 1))
common_units = units[units.common_valid.astype(str).str.lower().isin(["true", "1"])]
display(common_units.groupby("roi").agg(n_1km=("flag12", "size"), flag12_positive=("flag12", "sum"), flag15_positive=("flag15", "sum")))
display(agreement[["roi", "flag", "model", "threshold", "n", "tp", "fn", "fp", "tn", "recall", "specificity"]].round(4))
if not common_units[["flag12", "flag15"]].to_numpy().any():
    print("共同有效单元没有雪标志阳性；召回率为空，当前统计仅说明阴性一致性。")

## 5 整体误差与混合雪误差

WorldView 参考提供可见雪覆盖面积比例。每场数据先取四种方法共同有效的像元，再分别计算无雪、混合雪和近纯雪误差。比较总体与分层结果时，同时观察样本数量。

In [ ]:
worldview = t.worldview_metrics()
frozen = pd.read_csv(t.demo_directory() / "stratified_metrics.csv")
comparison = worldview.merge(frozen[["scene", "method", "stratum", "common_n", "common_rmse"]], on=["scene", "method", "stratum"], validate="one_to_one")
assert len(comparison) == 32
np.testing.assert_array_equal(comparison.n, comparison.common_n)
np.testing.assert_allclose(comparison.rmse, comparison.common_rmse, rtol=1e-5, atol=5e-7, equal_nan=True)
display(worldview.query("method in ['fixed200_six_bands', 'multi_six_bands']").round(6))
figure = t.plot_worldview_strata(worldview)
display(figure); plt.close(figure)

## 6 尺度处理的对应关系

固定字典尺度对照区分“仅光谱退化”“仅空间聚合”“二者同时进行”以及两种处理顺序。下面从修正后的共同有效表重新计算配对差异。参照值是高分辨率模型面积平均。

In [ ]:
control = pd.read_csv(t.demo_directory() / "same_dictionary_control_corrected.csv")
required = ["reference_snow", "fine_modis_aggregated_snow", "spatial_only_snow", "combined_snow"]
assert all(name in control.columns for name in required)
rows = []
for column in ["spatial_only_snow", "combined_snow"]:
    valid = (control["reference_valid_area_fraction"] >= .95) & (control["coverage"] >= .95)
    rows.append({"comparison": column, **t.difference_metrics(control.loc[valid, "reference_snow"], control.loc[valid, column])})
display(pd.DataFrame(rows).round(6))

## 回到研究问题

固定半径扫描回答雪端元变化如何影响同一观测；影像雪谱分支检查谱库选择；MODIS 账本检查验证样本如何被联合筛选；WorldView 分层表则把总体误差拆回具体雪况。把样本单位、数量、共同掩膜和误差放在一起，才能判断某种方法在哪些条件下适用。

练习请另开 cell：把面积阈值设为 0.8，调用 `t.validation_ledger(units, minimum_coverage=.8)`，记录保留数与两类阳性数。讨论放宽有效面积时获得了多少样本，又引入了多少未覆盖面积。这里的代价可以量化，无需用一个总体正确率代替。

教学采用公开派生的 WorldView 参考（Stillinger & Bair，Zenodo 4031446，CC BY 4.0，已作裁剪、聚合和分层）。原始商业 WorldView 影像未纳入。完整遥感处理入口、来源与原文图表索引见案例首页；默认 Notebook 的输入均在仓库内。